# 0825_peace_008_type_expert_time_weight

`mapping.json`으로 각 `inspection_type`의 유효 피처만 선택하는 타입별 XGBoost 전문가 모델에 시간 기반 `sample_weight`를 추가한 실험입니다.

- 모델·피처·파라미터·시간 분할·임계값 선택 방식은 `0825_peace_004_type_expert_walk_forward`와 동일합니다.
- 차이는 각 Fold와 최종 학습에서 타입별 Train 구간 내부 시간순으로 `sample_weight`를 1.0에서 2.0까지 선형 증가시킨 점뿐입니다.
- 각 Fold에서 Calibration Recall 99% 조건으로 공통·타입별 임계값을 선택하고 바로 다음 미래 구간에서 평가합니다.
- Walk-forward 완료 후 0~70% Train, 70~80% 최종 임계값 선택, 80~100% Test 평가를 동일하게 수행합니다.
- 실행 과정은 콘솔과 `docs/peace/0825_peace_008_type_expert_time_weight.log`에 함께 기록합니다.


## 1. 설정, 경로 탐색과 실행 로그

노트북을 저장소 루트 또는 `notebooks/`에서 실행해도 같은 원본 파일과 로그 경로를 사용합니다.


In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_008_type_expert_time_weight"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80
TIME_WEIGHT_MIN = 1.0
TIME_WEIGHT_MAX = 2.0

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


def make_time_sample_weight(frame: pd.DataFrame, time_column: str = TIME_COLUMN):
    if len(frame) == 0:
        raise ValueError("빈 frame에는 시간 가중치를 만들 수 없습니다.")

    timestamps = pd.to_datetime(frame[time_column], utc=True)
    timestamp_ns = timestamps.astype("int64").to_numpy(dtype=np.float64, copy=False)
    min_ns = float(timestamp_ns.min())
    max_ns = float(timestamp_ns.max())

    if max_ns == min_ns:
        weights = np.full(len(frame), TIME_WEIGHT_MIN, dtype=np.float64)
        degenerate = True
    else:
        relative_position = (timestamp_ns - min_ns) / (max_ns - min_ns)
        weights = TIME_WEIGHT_MIN + relative_position * (TIME_WEIGHT_MAX - TIME_WEIGHT_MIN)
        degenerate = False

    summary = {
        "time_weight_min": float(weights.min()),
        "time_weight_max": float(weights.max()),
        "time_weight_mean": float(weights.mean()),
        "time_weight_degenerate": degenerate,
        "train_start_time": timestamps.min(),
        "train_end_time": timestamps.max(),
    }
    return weights, summary


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f time_weight_range=[%.1f, %.1f]",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
    TIME_WEIGHT_MIN,
    TIME_WEIGHT_MAX,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 02:47:15,516 | INFO | experiment=0825_peace_008_type_expert_time_weight


2026-08-25 02:47:15,517 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99 time_weight_range=[1.0, 2.0]


2026-08-25 02:47:15,517 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 02:47:15,518 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 02:47:15,518 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 02:47:15,518 | INFO | log_file=docs/peace/0825_peace_008_type_expert_time_weight.log


log saved to: docs/peace/0825_peace_008_type_expert_time_weight.log


## 2. 원본 데이터와 매핑 검증

첫 번째 익명 인덱스 열은 `record_id`로 이름만 바꾸며 원본 파일은 수정하지 않습니다. 중복 제거는 원인 확인 전 데이터 의미를 바꿀 수 있어 이번 베이스라인에서 수행하지 않습니다.


In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 02:47:19,981 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 유효 피처

각 전문가 모델은 공통 `meta_feat1~4`와 `mapping.json`에 명시된 해당 타입의 `inspection_feat`만 사용합니다. 타입 분리 후 상수인 `inspection_type`과 식별자·시간·타깃은 입력에서 제외합니다.


In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 02:47:19,991 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 시간순 Train/Validation/Test 분할

전체 행의 누적 비율에 가장 가까운 timestamp 그룹 끝을 경계로 사용합니다. 같은 timestamp 그룹은 서로 다른 구간에 들어가지 않습니다.

- 0~70%: Train
- 70~80%: Validation
- 80~100%: 최종 Test


In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "fixed_test_threshold": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test     False
threshold_selected_on_test    False
fixed_test_threshold            0.5
Name: evaluation_policy, dtype: object

2026-08-25 02:47:20,377 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 02:47:20,377 | INFO | test_policy model_selection=False threshold=0.50


## 5. Peace 실험과 동일한 평가 지표

PR-AUC, ROC-AUC, Accuracy, Precision, Recall, F1, TP/FN/FP/TN, False Call Reduction을 계산합니다. Threshold 0.5는 베이스라인 비교용이며 운영 임계값이 아닙니다.


In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


2026-08-25 02:47:20,399 | INFO | threshold_selector_unit_test=PASS


## 6. 3-Fold Expanding Walk-forward 검증

첫 70% 개발 구간 안에서 Train을 누적 확장합니다. 각 Fold의 Calibration에서 임계값을 선택하고, 그 임계값을 바로 다음 미래 Evaluation에 고정 적용합니다.

각 타입 모델 학습에서는 해당 Fold Train 내부 시간순 위치만 사용해 `sample_weight`를 1.0에서 2.0까지 선형 증가시킵니다.

| Fold | Train | Calibration | Evaluation |
|---|---:|---:|---:|
| Fold 1 | 0~30% | 30~40% | 40~50% |
| Fold 2 | 0~40% | 40~50% | 50~60% |
| Fold 3 | 0~50% | 50~60% | 60~70% |

Calibration과 Evaluation은 모델 학습에 사용하지 않으며, Evaluation은 임계값 선택에도 사용하지 않습니다.


In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 02:47:21,075 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

In [7]:
def fit_type_experts_for_fold(train_frame, calibration_frame, evaluation_frame, fold_name):
    calibration_probability = pd.Series(
        np.nan, index=calibration_frame.index, dtype="float64"
    )
    evaluation_probability = pd.Series(
        np.nan, index=evaluation_frame.index, dtype="float64"
    )
    training_rows = []

    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type]
        type_calibration = calibration_frame.loc[
            calibration_frame[TYPE_COLUMN] == inspection_type
        ]
        type_evaluation = evaluation_frame.loc[
            evaluation_frame[TYPE_COLUMN] == inspection_type
        ]
        y_train = type_train[TARGET].astype("int8")

        assert len(type_train) > 0
        assert len(type_calibration) > 0
        assert len(type_evaluation) > 0
        assert y_train.nunique() == 2
        assert type_calibration[TARGET].nunique() == 2

        sample_weight, weight_summary = make_time_sample_weight(type_train)
        logger.info(
            "walk_forward_fit_start fold=%s type=%d train_rows=%d train_positive=%d calibration_rows=%d calibration_positive=%d evaluation_rows=%d evaluation_positive=%d weight_min=%.6f weight_max=%.6f weight_mean=%.6f weight_degenerate=%s",
            fold_name,
            inspection_type,
            len(type_train),
            int(y_train.sum()),
            len(type_calibration),
            int(type_calibration[TARGET].sum()),
            len(type_evaluation),
            int(type_evaluation[TARGET].sum()),
            weight_summary["time_weight_min"],
            weight_summary["time_weight_max"],
            weight_summary["time_weight_mean"],
            weight_summary["time_weight_degenerate"],
        )

        preprocessor = make_preprocessor(feature_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        X_calibration = preprocessor.transform(type_calibration[feature_columns])
        X_evaluation = preprocessor.transform(type_evaluation[feature_columns])

        model = XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
        calibration_probability.loc[type_calibration.index] = model.predict_proba(
            X_calibration
        )[:, 1]
        evaluation_probability.loc[type_evaluation.index] = model.predict_proba(
            X_evaluation
        )[:, 1]

        training_rows.append(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "train_rows": len(type_train),
                "train_positive": int(y_train.sum()),
                "calibration_rows": len(type_calibration),
                "calibration_positive": int(type_calibration[TARGET].sum()),
                "evaluation_rows": len(type_evaluation),
                "evaluation_positive": int(type_evaluation[TARGET].sum()),
                "raw_features": len(feature_columns),
                "encoded_features": X_train.shape[1],
                **weight_summary,
            }
        )
        logger.info("walk_forward_fit_done fold=%s type=%d", fold_name, inspection_type)
        del preprocessor, model, X_train, X_calibration, X_evaluation, sample_weight
        gc.collect()

    assert calibration_probability.notna().all()
    assert evaluation_probability.notna().all()
    return calibration_probability, evaluation_probability, training_rows


walk_forward_threshold_rows = []
walk_forward_metric_rows = []
walk_forward_type_evaluation_rows = []
walk_forward_training_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    segments = walk_forward_segments[fold_name]
    calibration_frame = segments["calibration"]
    evaluation_frame = segments["evaluation"]
    calibration_probability, evaluation_probability, training_rows = (
        fit_type_experts_for_fold(
            segments["train"], calibration_frame, evaluation_frame, fold_name
        )
    )
    walk_forward_training_rows.extend(training_rows)

    global_selection = select_threshold(
        calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL
    )
    walk_forward_threshold_rows.append(
        {"fold": fold_name, "scope": "global", **global_selection}
    )

    thresholds_by_type_fold = {}
    type_evaluation_prediction = pd.Series(
        np.nan, index=evaluation_frame.index, dtype="float64"
    )
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[
            calibration_frame[TYPE_COLUMN] == inspection_type
        ]
        type_calibration_probability = calibration_probability.loc[
            type_calibration.index
        ]
        selection = select_threshold(
            type_calibration[TARGET],
            type_calibration_probability,
            min_recall=MIN_RECALL,
        )
        threshold = selection["threshold"]
        thresholds_by_type_fold[inspection_type] = threshold
        walk_forward_threshold_rows.append(
            {
                "fold": fold_name,
                "scope": f"type_{inspection_type}",
                **selection,
            }
        )

        type_evaluation = evaluation_frame.loc[
            evaluation_frame[TYPE_COLUMN] == inspection_type
        ]
        type_evaluation_probability = evaluation_probability.loc[type_evaluation.index]
        type_prediction = (type_evaluation_probability >= threshold).astype("int8")
        type_evaluation_prediction.loc[type_evaluation.index] = type_prediction
        type_metrics = evaluate_predictions(
            type_evaluation[TARGET], type_prediction, type_evaluation_probability
        )
        type_metrics.update(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "threshold": threshold,
            }
        )
        walk_forward_type_evaluation_rows.append(type_metrics)

    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(
            evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD
        ),
        "global_threshold": evaluate_probabilities(
            evaluation_frame[TARGET],
            evaluation_probability,
            global_selection["threshold"],
        ),
        "type_specific_thresholds": evaluate_predictions(
            evaluation_frame[TARGET],
            type_evaluation_prediction,
            evaluation_probability,
        ),
    }
    for strategy, metrics in strategy_metrics.items():
        walk_forward_metric_rows.append(
            {"fold": fold_name, "strategy": strategy, **metrics}
        )

    logger.info(
        "walk_forward_fold_done fold=%s global_threshold=%.8f metrics=%s",
        fold_name,
        global_selection["threshold"],
        strategy_metrics,
    )

walk_forward_threshold_summary = pd.DataFrame(walk_forward_threshold_rows).set_index(
    ["fold", "scope"]
)
walk_forward_evaluation_metrics = pd.DataFrame(walk_forward_metric_rows).set_index(
    ["fold", "strategy"]
)
walk_forward_type_evaluation = pd.DataFrame(
    walk_forward_type_evaluation_rows
).set_index(["fold", "inspection_type"])
walk_forward_training_summary = pd.DataFrame(walk_forward_training_rows).set_index(
    ["fold", "inspection_type"]
)


2026-08-25 02:47:21,125 | INFO | walk_forward_fit_start fold=fold_1 type=0 train_rows=28277 train_positive=32 calibration_rows=8408 calibration_positive=11 evaluation_rows=6496 evaluation_positive=50 weight_min=1.000000 weight_max=2.000000 weight_mean=1.516054 weight_degenerate=False


2026-08-25 02:47:21,579 | INFO | walk_forward_fit_done fold=fold_1 type=0


2026-08-25 02:47:21,611 | INFO | walk_forward_fit_start fold=fold_1 type=1 train_rows=22698 train_positive=269 calibration_rows=3868 calibration_positive=20 evaluation_rows=2618 evaluation_positive=186 weight_min=1.000000 weight_max=2.000000 weight_mean=1.475024 weight_degenerate=False


2026-08-25 02:47:22,194 | INFO | walk_forward_fit_done fold=fold_1 type=1


2026-08-25 02:47:22,231 | INFO | walk_forward_fit_start fold=fold_1 type=2 train_rows=42288 train_positive=408 calibration_rows=16448 calibration_positive=92 evaluation_rows=18964 evaluation_positive=49 weight_min=1.000000 weight_max=2.000000 weight_mean=1.593579 weight_degenerate=False


...

2026-08-25 02:47:31,587 | INFO | walk_forward_fit_start fold=fold_3 type=3 train_rows=67320 train_positive=622 calibration_rows=20747 calibration_positive=23 evaluation_rows=12673 evaluation_positive=3 weight_min=1.000000 weight_max=2.000000 weight_mean=1.525556 weight_degenerate=False


2026-08-25 02:47:32,572 | INFO | walk_forward_fit_done fold=fold_3 type=3


2026-08-25 02:47:32,600 | INFO | walk_forward_fit_start fold=fold_3 type=4 train_rows=2771 train_positive=10 calibration_rows=698 calibration_positive=3 evaluation_rows=344 evaluation_positive=0 weight_min=1.000000 weight_max=2.000000 weight_mean=1.526082 weight_degenerate=False


2026-08-25 02:47:32,692 | INFO | walk_forward_fit_done fold=fold_3 type=4


2026-08-25 02:47:32,970 | INFO | walk_forward_fold_done fold=fold_3 global_threshold=0.00009794 metrics={'fixed_0.5': {'rows': 43853, 'positive_samples': 39, 'tn': 43781, 'fp': 33, 'fn': 37, 'tp': 2, 'accuracy': 0.9984037580097143, 'precision': 0.05714285714285714, 'recall': 0.05128205128205128, 'false_call_reduction': 0.9992468160861825, 'f1': 0.05405405405405406, 'roc_auc': 0.9261168716708041, 'pr_auc': 0.028369887766536178}, 'global_threshold': {'rows': 43853, 'positive_samples': 39, 'tn': 6297, 'fp': 37517, 'fn': 0, 'tp': 39, 'accuracy': 0.14448270357786241, 'precision': 0.0010384492491213122, 'recall': 1.0, 'false_call_reduction': 0.14372118500935774, 'f1': 0.0020747439819124884, 'roc_auc': 0.9261168716708041, 'pr_auc': 0.028369887766536178}, 'type_specific_thresholds': {'rows': 43853, 'positive_samples': 39, 'tn': 9132, 'fp': 34682, 'fn': 0, 'tp': 39, 'accuracy': 0.20913050418443435, 'precision': 0.0011232395380317388, 'recall': 1.0, 'false_call_reduction': 0.20842653033277034, '

## 7. Walk-forward 미래 Evaluation 결과

공통·타입별 임계값은 각 Fold의 Calibration에서만 선택됐습니다. 아래 지표는 임계값 선택에 사용하지 않은 바로 다음 미래 Evaluation 결과입니다.

In [8]:
display(
    walk_forward_threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]
    ]
)
display(
    walk_forward_evaluation_metrics[
        [
            "positive_samples",
            "pr_auc",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(
    walk_forward_type_evaluation[
        [
            "threshold",
            "positive_samples",
            "pr_auc",
            "recall",
            "false_call_reduction",
            "tp",
            "fn",
        ]
    ]
)

walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index()
    .groupby("strategy")
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"),
        total_fn=("fn", "sum"),
    )
)
display(walk_forward_strategy_summary)
display(walk_forward_training_summary)
logger.info(
    "walk_forward_strategy_summary=%s",
    walk_forward_strategy_summary.to_dict(orient="index"),
)


threshold  positive_samples    recall  false_call_reduction  \
fold   scope                                                                 
fold_1 global   0.000018               200  0.990000              0.013203   
       type_0   0.002591                11  1.000000              0.399905   
       type_1   0.000700                20  1.000000              0.223753   
       type_2   0.000009                92  1.000000              0.003913   
       type_3   0.000409                73  1.000000              0.421372   
       type_4   0.002373                 4  1.000000              0.000000   
fold_2 global   0.000455               326  0.990798              0.350917   
       type_0   0.001026                50  1.000000              0.648774   
       type_1   0.000463               186  0.994624              0.240954   
       type_2   0.000177                49  1.000000              0.275496   
       type_3   0.009036                39  1.000000              0.705667   
       type_4   0.003460                 2  1.000000              0.000000   
fold_3 global   0.000098               152  0.993421              0.254956   
       type_0   0.000134                14  1.000000              0.435514   
       type_1   0.000528                80  1.000000              0.116528   
       type_2   0.000063                32  1.000000              0.081131   
       type_3   0.000173                23  1.000000              0.542270   
       type_4   0.003891                 3  1.000000              0.000000   

                tp  fn  
fold   scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
fold   strategy                                                          
fold_1 fixed_0.5                              326  0.151108   0.267544   
       global_threshold                       326  0.151108   0.007472   
       type_specific_thresholds               326  0.151108   0.008997   
fold_2 fixed_0.5                              152  0.029274   0.030928   
       global_threshold                       152  0.029274   0.005559   
       type_specific_thresholds               152  0.029274   0.007573   
fold_3 fixed_0.5                               39  0.028370   0.057143   
       global_threshold                        39  0.028370   0.001038   
       type_specific_thresholds                39  0.028370   0.001123   

                                   recall  false_call_reduction        f1  \
fold   strategy                                                             
fold_1 fixed_0.5                 0.187117              0.996180  0.220217   
       global_threshold          1.000000              0.009356  0.014833   
       type_specific_thresholds  0.987730              0.188681  0.017832   
fold_2 fixed_0.5                 0.039474              0.995731  0.034682   
       global_threshold          0.927632              0.427251  0.011053   
       type_specific_thresholds  0.822368              0.628023  0.015009   
fold_3 fixed_0.5                 0.051282              0.999247  0.054054   
       global_threshold          1.000000              0.143721  0.002075   
       type_specific_thresholds  1.000000              0.208427  0.002244   

                                  tp   fn     fp     tn  
fold   strategy                                          
fold_1 fixed_0.5                  61  265    167  43547  
       global_threshold          326    0  43305    409  
       type_specific_thresholds  322    4  35466   8248  
fold_2 fixed_0.5                   6  146    188  43847  
       global_threshold          141   11  25221  18814  
       type_specific_thresholds  125   27  16380  27655  
fold_3 fixed_0.5                   2   37     33  43781  
       global_threshold           39    0  37517   6297  
       type_specific_thresholds   39    0  34682   9132

threshold  positive_samples    pr_auc    recall  \
fold   inspection_type                                                    
fold_1 0                 0.002591                50  0.099411  0.960000   
       1                 0.000700               186  0.234462  0.989247   
       2                 0.000009                49  0.330674  1.000000   
       3                 0.000409                39  0.570201  1.000000   
       4                 0.002373                 2  0.006154  1.000000   
fold_2 0                 0.001026                14  0.003308  0.500000   
       1                 0.000463                80  0.242711  1.000000   
       2                 0.000177                32  0.029054  0.937500   
       3                 0.009036                23  0.001675  0.217391   
       4                 0.003460                 3  0.004298  1.000000   
fold_3 0                 0.000134                 4  0.007458  1.000000   
       1                 0.000528                25  0.028252  1.000000   
       2                 0.000063                 7  0.023613  1.000000   
       3                 0.000173                 3  0.336073  1.000000   
       4                 0.003891                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
fold   inspection_type                                 
fold_1 0                            0.678715   48   2  
       1                            0.181743  184   2  
       2                            0.001745   49   0  
       3                            0.217848   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.810501    7   7  
       1                            0.493020   80   0  
       2                            0.124799   30   2  
       3                            0.813598    5  18  
       4                            0.000000    3   0  
fold_3 0                            0.177642    4   0  
       1                            0.101757   25   0  
       2                            0.036852    7   0  
       3                            0.472770    3   0  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.069584,0.092624,0.039474,0,0.997052,0.995731,69,448
global_threshold,3,0.069584,0.975877,0.927632,2,0.193443,0.009356,506,11
type_specific_thresholds,3,0.069584,0.936699,0.822368,1,0.341710,0.188681,486,31


train_rows  train_positive  calibration_rows  \
fold   inspection_type                                                 
fold_1 0                     28277              32              8408   
       1                     22698             269              3868   
       2                     42288             408             16448   
       3                     37264             510             14419   
       4                      1610               4               836   
fold_2 0                     36685              43              6496   
       1                     26566             289              2618   
       2                     58736             500             18964   
       3                     51683             583             15637   
       4                      2446               8               325   
fold_3 0                     43181              93              8985   
       1                     29184             475              5023   
       2                     77700             549              8734   
       3                     67320             622             20747   
       4                      2771              10               698   

                        calibration_positive  evaluation_rows  \
fold   inspection_type                                          
fold_1 0                                  11             6496   
       1                                  20             2618   
       2                                  92            18964   
       3                                  73            15637   
       4                                   4              325   
fold_2 0                                  50             8985   
       1                                 186             5023   
       2                                  49             8734   
       3                                  39            20747   
       4                                   2              698   
fold_3 0                                  14            12107   
       1                                  80             4693   
       2                                  32            14036   
       3                                  23            12673   
       4                                   3              344   

                        evaluation_positive  raw_features  encoded_features  \
fold   inspection_type                                                        
fold_1 0                                 50            48                80   
       1                                186            56               106   
       2                                 49            69               114   
       3                                 39            69               107   
       4                                  2            25                47   
fold_2 0                                 14            48                82   
       1                                 80            56               110   
       2                                 32            69               114   
       3                                 23            69               107   
       4                                  3            25                47   
fold_3 0                                  4            48                84   
       1                                 25            56               111   
       2                                  7            69               115   
       3                                  3            69               108   
       4                                  0            25                50   

                        time_weight_min  time_weight_max  time_weight_mean  \
fold   inspection_type                                                       
fold_1 0                            1.0              2.0          1.516054   
       1                            1.0              2.0          1.475024   
       2                            1

2026-08-25 02:47:32,995 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.06958379976550501, 'mean_recall': 0.0926240999699185, 'min_recall': 0.039473684210526314, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.997052399488425, 'min_false_call_reduction': 0.9957306687861928, 'total_tp': 69, 'total_fn': 448}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.06958379976550501, 'mean_recall': 0.9758771929824562, 'min_recall': 0.9276315789473685, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.193442835204225, 'min_false_call_reduction': 0.009356270302420278, 'total_tp': 506, 'total_fn': 11}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.06958379976550501, 'mean_recall': 0.9366994941341082, 'min_recall': 0.8223684210526315, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.3417102218321945, 'min_false_call_reduction': 0.18868097177105733, 'total_tp': 486, 'total_fn': 31}}


## 8. 최종 타입별 전문가 모델 5개 학습

각 타입에서 전처리기는 Train에만 `fit`합니다. 최종 0~70% Train 내부 시간순으로 `sample_weight`를 1.0에서 2.0까지 선형 증가시켜 학습하며, 클래스 가중치와 리샘플링은 적용하지 않습니다.


In [9]:
pooled_probability = pd.Series(np.nan, index=validation_df.index, dtype="float64")
models_by_type = {}
preprocessors_by_type = {}
type_metric_rows = []
training_rows = []

for inspection_type in inspection_types:
    feature_columns = feature_columns_by_type[inspection_type]
    type_train = train_df.loc[train_df[TYPE_COLUMN] == inspection_type]
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    y_train = type_train[TARGET].astype("int8")
    y_validation = type_validation[TARGET].astype("int8")

    assert len(type_train) > 0 and len(type_validation) > 0
    assert y_train.nunique() == 2, f"type={inspection_type} Train에 두 클래스가 없습니다."
    sample_weight, weight_summary = make_time_sample_weight(type_train)
    logger.info(
        "model_fit_start type=%d train_rows=%d train_positive=%d valid_rows=%d valid_positive=%d raw_features=%d weight_min=%.6f weight_max=%.6f weight_mean=%.6f weight_degenerate=%s",
        inspection_type,
        len(type_train),
        int(y_train.sum()),
        len(type_validation),
        int(y_validation.sum()),
        len(feature_columns),
        weight_summary["time_weight_min"],
        weight_summary["time_weight_max"],
        weight_summary["time_weight_mean"],
        weight_summary["time_weight_degenerate"],
    )

    preprocessor = make_preprocessor(feature_columns)
    X_train = preprocessor.fit_transform(type_train[feature_columns])
    X_validation = preprocessor.transform(type_validation[feature_columns])
    assert np.isfinite(X_train.data if hasattr(X_train, "data") else X_train).all()
    assert np.isfinite(X_validation.data if hasattr(X_validation, "data") else X_validation).all()

    model = XGBClassifier(**XGB_PARAMS)
    model.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
    probability = model.predict_proba(X_validation)[:, 1]
    pooled_probability.loc[type_validation.index] = probability

    metrics = evaluate_probabilities(y_validation, probability)
    metrics["inspection_type"] = inspection_type
    type_metric_rows.append(metrics)
    training_rows.append(
        {
            "inspection_type": inspection_type,
            "train_rows": len(type_train),
            "train_positive": int(y_train.sum()),
            "validation_rows": len(type_validation),
            "validation_positive": int(y_validation.sum()),
            "raw_features": len(feature_columns),
            "encoded_features": X_train.shape[1],
            "trees": model.n_estimators,
            **weight_summary,
        }
    )
    models_by_type[inspection_type] = model
    preprocessors_by_type[inspection_type] = preprocessor
    logger.info(
        "model_fit_done type=%d pr_auc=%.6f recall=%.6f fcr=%.6f tp=%d fn=%d",
        inspection_type,
        metrics["pr_auc"],
        metrics["recall"],
        metrics["false_call_reduction"],
        metrics["tp"],
        metrics["fn"],
    )
    del X_train, X_validation, probability, sample_weight
    gc.collect()

assert pooled_probability.notna().all()
pooled_metrics = pd.Series(
    evaluate_probabilities(validation_df[TARGET], pooled_probability),
    name="type_expert_validation",
)
type_metrics = pd.DataFrame(type_metric_rows).set_index("inspection_type")
training_summary = pd.DataFrame(training_rows).set_index("inspection_type")
logger.info("pooled_validation_metrics=%s", pooled_metrics.to_dict())


2026-08-25 02:47:33,051 | INFO | model_fit_start type=0 train_rows=64273 train_positive=111 valid_rows=13289 valid_positive=12 raw_features=48 weight_min=1.000000 weight_max=2.000000 weight_mean=1.571337 weight_degenerate=False


2026-08-25 02:47:33,828 | INFO | model_fit_done type=0 pr_auc=0.005336 recall=0.000000 fcr=0.999925 tp=0 fn=12


2026-08-25 02:47:33,867 | INFO | model_fit_start type=1 train_rows=38900 train_positive=580 valid_rows=6422 valid_positive=224 raw_features=56 weight_min=1.000000 weight_max=2.000000 weight_mean=1.482090 weight_degenerate=False


2026-08-25 02:47:34,555 | INFO | model_fit_done type=1 pr_auc=0.714095 recall=0.642857 fcr=0.988706 tp=144 fn=80


2026-08-25 02:47:34,598 | INFO | model_fit_start type=2 train_rows=100470 train_positive=588 valid_rows=7161 valid_positive=27 raw_features=69 weight_min=1.000000 weight_max=2.000000 weight_mean=1.571788 weight_degenerate=False


...

2026-08-25 02:47:36,061 | INFO | model_fit_start type=3 train_rows=100740 train_positive=648 valid_rows=16252 valid_positive=21 raw_features=69 weight_min=1.000000 weight_max=2.000000 weight_mean=1.580274 weight_degenerate=False


2026-08-25 02:47:37,386 | INFO | model_fit_done type=3 pr_auc=0.064377 recall=0.047619 fcr=0.998521 tp=1 fn=20


2026-08-25 02:47:37,415 | INFO | model_fit_start type=4 train_rows=3813 train_positive=13 valid_rows=902 valid_positive=73 raw_features=25 weight_min=1.000000 weight_max=2.000000 weight_mean=1.556784 weight_degenerate=False


2026-08-25 02:47:37,521 | INFO | model_fit_done type=4 pr_auc=0.080931 recall=0.000000 fcr=1.000000 tp=0 fn=73


2026-08-25 02:47:37,581 | INFO | pooled_validation_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43571.0, 'fp': 98.0, 'fn': 203.0, 'tp': 154.0, 'accuracy': 0.993163130877209, 'precision': 0.6111111111111112, 'recall': 0.43137254901960786, 'false_call_reduction': 0.9977558451075134, 'f1': 0.5057471264367817, 'roc_auc': 0.9491792182764242, 'pr_auc': 0.47999695939966774}


## 9. 최종 Validation 결과


In [10]:
count_columns = ["rows", "positive_samples", "tn", "fp", "fn", "tp"]
type_metrics[count_columns] = type_metrics[count_columns].astype("int64")
display(pooled_metrics)
display(
    type_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(training_summary)


rows                    44026.000000
positive_samples          357.000000
tn                      43571.000000
fp                         98.000000
fn                        203.000000
tp                        154.000000
accuracy                    0.993163
precision                   0.611111
recall                      0.431373
false_call_reduction        0.997756
f1                          0.505747
roc_auc                     0.949179
pr_auc                      0.479997
Name: type_expert_validation, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,13289,12,0.005336,0.856952,0.999022,0.000000,0.000000,0.999925,0.000000,0,12,1,13276
1,6422,224,0.714095,0.963450,0.976643,0.672897,0.642857,0.988706,0.657534,144,80,70,6128
2,7161,27,0.401027,0.918676,0.997067,0.750000,0.333333,0.999579,0.461538,9,18,3,7131
3,16252,21,0.064377,0.931058,0.997293,0.040000,0.047619,0.998521,0.043478,1,20,24,16207
4,902,73,0.080931,0.500000,0.919069,0.000000,0.000000,1.000000,0.000000,0,73,0,829


,train_rows,train_positive,validation_rows,validation_positive,raw_features,encoded_features,trees,time_weight_min,time_weight_max,time_weight_mean,time_weight_degenerate,train_start_time,train_end_time
inspection_type,,,,,,,,,,,,,
0,64273,111,13289,12,48,88,400,1.0,2.0,1.571337,False,1970-06-23 03:58:55+00:00,1970-10-05 00:29:00+00:00
1,38900,580,6422,224,56,113,400,1.0,2.0,1.482090,False,1970-06-23 05:00:03+00:00,1970-10-05 00:29:59+00:00
2,100470,588,7161,27,69,117,400,1.0,2.0,1.571788,False,1970-06-23 04:01:30+00:00,1970-10-05 00:29:59+00:00
3,100740,648,16252,21,69,109,400,1.0,2.0,1.580274,False,1970-06-23 04:00:54+00:00,1970-10-05 00:29:59+00:00
4,3813,13,902,73,25,53,400,1.0,2.0,1.556784,False,1970-06-24 12:16:18+00:00,1970-10-05 00:12:57+00:00


## 10. 최종 Validation에서 공통·타입별 임계값 선택

Test를 사용하지 않고 Validation Recall 99% 이상을 만족하는 후보 중 False Call Reduction이 최대인 임계값을 선택합니다. 동률이면 Recall, 다시 동률이면 threshold가 높은 후보를 선택합니다.

In [11]:
global_threshold_selection = select_threshold(
    validation_df[TARGET], pooled_probability, min_recall=MIN_RECALL
)

thresholds_by_type = {}
type_threshold_rows = []
type_validation_prediction = pd.Series(np.nan, index=validation_df.index, dtype="float64")

for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_probability = pooled_probability.loc[type_validation.index]
    selection = select_threshold(
        type_validation[TARGET], type_probability, min_recall=MIN_RECALL
    )
    thresholds_by_type[inspection_type] = selection["threshold"]
    selection["inspection_type"] = inspection_type
    type_threshold_rows.append(selection)
    type_validation_prediction.loc[type_validation.index] = (
        type_probability >= selection["threshold"]
    ).astype("int8")

type_threshold_selection = pd.DataFrame(type_threshold_rows).set_index("inspection_type")
global_validation_metrics = pd.Series(
    evaluate_probabilities(
        validation_df[TARGET],
        pooled_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_specific_validation_metrics = pd.Series(
    evaluate_predictions(
        validation_df[TARGET],
        type_validation_prediction,
        pooled_probability,
    ),
    name="type_specific_thresholds",
)

validation_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": pooled_metrics,
        "global_threshold": global_validation_metrics,
        "type_specific_thresholds": type_specific_validation_metrics,
    }
).T

threshold_summary = pd.concat(
    [
        pd.DataFrame(
            [{"scope": "global", **global_threshold_selection}]
        ).set_index("scope"),
        type_threshold_selection.rename_axis("scope"),
    ],
    axis=0,
)

display(
    threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(
    validation_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
logger.info("global_threshold_selection=%s", global_threshold_selection)
logger.info("type_threshold_selection=%s", type_threshold_selection.to_dict(orient="index"))
logger.info("validation_strategy_metrics=%s", validation_strategy_metrics.to_dict(orient="index"))

,threshold,positive_samples,recall,false_call_reduction,tp,fn,fp,tn
scope,,,,,,,,
global,0.000486,357,0.991597,0.618585,354,3,16656,27013
0,0.000152,12,1.000000,0.411463,12,0,7814,5463
1,0.000917,224,0.991071,0.503388,222,2,3078,3120
2,0.000118,27,1.000000,0.416176,27,0,4165,2969
3,0.000150,21,1.000000,0.449633,21,0,8933,7298
4,0.003480,73,1.000000,0.000000,73,0,829,0


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.479997,0.611111,0.431373,0.997756,0.505747,154.0,203.0,98.0,43571.0
global_threshold,0.479997,0.020811,0.991597,0.618585,0.040767,354.0,3.0,16656.0,27013.0
type_specific_thresholds,0.479997,0.014102,0.994398,0.431656,0.027809,355.0,2.0,24819.0,18850.0


2026-08-25 02:47:37,784 | INFO | global_threshold_selection={'threshold': 0.0004856879240833223, 'min_recall': 0.99, 'rows': 44026, 'positive_samples': 357, 'tn': 27013, 'fp': 16656, 'fn': 3, 'tp': 354, 'accuracy': 0.6216099577522373, 'precision': 0.020811287477954146, 'recall': 0.9915966386554622, 'false_call_reduction': 0.6185852664361446, 'f1': 0.040766971843150805, 'roc_auc': 0.9491792182764242, 'pr_auc': 0.47999695939966774}


2026-08-25 02:47:37,784 | INFO | type_threshold_selection={0: {'threshold': 0.00015194129082374275, 'min_recall': 0.99, 'rows': 13289, 'positive_samples': 12, 'tn': 5463, 'fp': 7814, 'fn': 0, 'tp': 12, 'accuracy': 0.41199488298592823, 'precision': 0.001533350370559673, 'recall': 1.0, 'false_call_reduction': 0.41146343300444377, 'f1': 0.0030620056136769582, 'roc_auc': 0.856951871657754, 'pr_auc': 0.005336250353858211}, 1: {'threshold': 0.0009171736892312765, 'min_recall': 0.99, 'rows': 6422, 'positive_samples': 224, 'tn': 3120, 'fp': 3078, 'fn': 2, 'tp': 222, 'accuracy': 0.5203986297103707, 'precision': 0.06727272727272728, 'recall': 0.9910714285714286, 'false_call_reduction': 0.5033881897386253, 'f1': 0.12599318955732122, 'roc_auc': 0.9634498311667359, 'pr_auc': 0.7140950095884786}, 2: {'threshold': 0.00011768970580305904, 'min_recall': 0.99, 'rows': 7161, 'positive_samples': 27, 'tn': 2969, 'fp': 4165, 'fn': 0, 'tp': 27, 'accuracy': 0.41837732160312807, 'precision': 0.0064408396946564

2026-08-25 02:47:37,785 | INFO | validation_strategy_metrics={'fixed_0.5': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43571.0, 'fp': 98.0, 'fn': 203.0, 'tp': 154.0, 'accuracy': 0.993163130877209, 'precision': 0.6111111111111112, 'recall': 0.43137254901960786, 'false_call_reduction': 0.9977558451075134, 'f1': 0.5057471264367817, 'roc_auc': 0.9491792182764242, 'pr_auc': 0.47999695939966774}, 'global_threshold': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 27013.0, 'fp': 16656.0, 'fn': 3.0, 'tp': 354.0, 'accuracy': 0.6216099577522373, 'precision': 0.020811287477954146, 'recall': 0.9915966386554622, 'false_call_reduction': 0.6185852664361446, 'f1': 0.040766971843150805, 'roc_auc': 0.9491792182764242, 'pr_auc': 0.47999695939966774}, 'type_specific_thresholds': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 18850.0, 'fp': 24819.0, 'fn': 2.0, 'tp': 355.0, 'accuracy': 0.4362195066551583, 'precision': 0.014101851116231032, 'recall': 0.9943977591036415, 'false_call_reduction': 

## 11. 고정 모델의 최종 Test 추론

Validation 결과를 확인한 뒤 모델·피처·파라미터와 Validation에서 선택한 threshold를 변경하지 않고 마지막 20% Test를 한 번 추론합니다.

In [12]:
test_probability = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_test_metric_rows = []

for inspection_type in inspection_types:
    feature_columns = feature_columns_by_type[inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    preprocessor = preprocessors_by_type[inspection_type]
    model = models_by_type[inspection_type]

    X_test = preprocessor.transform(type_test[feature_columns])
    probability = model.predict_proba(X_test)[:, 1]
    test_probability.loc[type_test.index] = probability

    metrics = evaluate_probabilities(type_test[TARGET], probability)
    metrics["inspection_type"] = inspection_type
    type_test_metric_rows.append(metrics)
    logger.info(
        "test_type_metrics type=%d pr_auc=%.6f recall=%.6f fcr=%.6f tp=%d fn=%d",
        inspection_type,
        metrics["pr_auc"],
        metrics["recall"],
        metrics["false_call_reduction"],
        metrics["tp"],
        metrics["fn"],
    )
    del X_test, probability
    gc.collect()

assert test_probability.notna().all()
test_metrics = pd.Series(
    evaluate_probabilities(test_df[TARGET], test_probability),
    name="type_expert_test",
)
type_test_metrics = pd.DataFrame(type_test_metric_rows).set_index("inspection_type")
type_test_metrics[count_columns] = type_test_metrics[count_columns].astype("int64")
fixed_test_metrics = test_metrics.copy()
fixed_test_metrics.name = "fixed_0.5"
global_test_metrics = pd.Series(
    evaluate_probabilities(
        test_df[TARGET],
        test_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_test_prediction = pd.Series(np.nan, index=test_df.index, dtype="float64")
type_selected_test_rows = []
for inspection_type in inspection_types:
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    type_probability = test_probability.loc[type_test.index]
    threshold = thresholds_by_type[inspection_type]
    type_prediction = (type_probability >= threshold).astype("int8")
    type_test_prediction.loc[type_test.index] = type_prediction
    metrics = evaluate_predictions(type_test[TARGET], type_prediction, type_probability)
    metrics.update({"inspection_type": inspection_type, "threshold": threshold})
    type_selected_test_rows.append(metrics)

type_specific_test_metrics = pd.Series(
    evaluate_predictions(test_df[TARGET], type_test_prediction, test_probability),
    name="type_specific_thresholds",
)
test_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": fixed_test_metrics,
        "global_threshold": global_test_metrics,
        "type_specific_thresholds": type_specific_test_metrics,
    }
).T
type_selected_test_metrics = pd.DataFrame(type_selected_test_rows).set_index("inspection_type")

logger.info("pooled_test_metrics_fixed_0.5=%s", fixed_test_metrics.to_dict())
logger.info("test_strategy_metrics=%s", test_strategy_metrics.to_dict(orient="index"))

display(
    test_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
display(
    type_selected_test_metrics[
        ["threshold", "positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(fixed_test_metrics)
display(
    type_test_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)

2026-08-25 02:47:37,855 | INFO | test_type_metrics type=0 pr_auc=0.032435 recall=0.005128 fcr=0.999534 tp=1 fn=194


2026-08-25 02:47:37,913 | INFO | test_type_metrics type=1 pr_auc=0.425714 recall=0.403101 fcr=0.974778 tp=312 fn=462


2026-08-25 02:47:37,994 | INFO | test_type_metrics type=2 pr_auc=0.652728 recall=0.214774 fcr=0.999798 tp=157 fn=574


2026-08-25 02:47:38,111 | INFO | test_type_metrics type=3 pr_auc=0.385687 recall=0.119281 fcr=0.998485 tp=73 fn=539


2026-08-25 02:47:38,140 | INFO | test_type_metrics type=4 pr_auc=0.017857 recall=0.000000 fcr=1.000000 tp=0 fn=13


...

2026-08-25 02:47:38,492 | INFO | test_strategy_metrics={'fixed_0.5': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 85370.0, 'fp': 357.0, 'fn': 1782.0, 'tp': 543.0, 'accuracy': 0.9757075364557307, 'precision': 0.6033333333333334, 'recall': 0.2335483870967742, 'false_call_reduction': 0.9958356177167054, 'f1': 0.33674418604651163, 'roc_auc': 0.8764222887583503, 'pr_auc': 0.37158692712359925}, 'global_threshold': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 42668.0, 'fp': 43059.0, 'fn': 213.0, 'tp': 2112.0, 'accuracy': 0.5085631217916685, 'precision': 0.046755661818423326, 'recall': 0.9083870967741936, 'false_call_reduction': 0.49771950494010053, 'f1': 0.08893380495199596, 'roc_auc': 0.8764222887583503, 'pr_auc': 0.37158692712359925}, 'type_specific_thresholds': {'rows': 88052.0, 'positive_samples': 2325.0, 'tn': 26796.0, 'fp': 58931.0, 'fn': 135.0, 'tp': 2190.0, 'accuracy': 0.3291918411847545, 'precision': 0.03583056559938483, 'recall': 0.9419354838709677, 'false_call_reducti

,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.371587,0.603333,0.233548,0.995836,0.336744,543.0,1782.0,357.0,85370.0
global_threshold,0.371587,0.046756,0.908387,0.497720,0.088934,2112.0,213.0,43059.0,42668.0
type_specific_thresholds,0.371587,0.035831,0.941935,0.312574,0.069035,2190.0,135.0,58931.0,26796.0


,threshold,positive_samples,pr_auc,precision,recall,false_call_reduction,tp,fn,fp,tn
inspection_type,,,,,,,,,,
0,0.000152,195,0.032435,0.013155,0.789744,0.401275,154,41,11553,7743
1,0.000917,774,0.425714,0.100451,0.979328,0.413665,758,16,6788,4789
2,0.000118,731,0.652728,0.042117,0.958960,0.195286,701,30,15943,3869
3,0.000150,612,0.385687,0.023024,0.921569,0.302823,564,48,23932,10395
4,0.003480,13,0.017857,0.017857,1.000000,0.000000,13,0,715,0


rows                    88052.000000
positive_samples         2325.000000
tn                      85370.000000
fp                        357.000000
fn                       1782.000000
tp                        543.000000
accuracy                    0.975708
precision                   0.603333
recall                      0.233548
false_call_reduction        0.995836
f1                          0.336744
roc_auc                     0.876422
pr_auc                      0.371587
Name: fixed_0.5, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,19491,195,0.032435,0.683713,0.989585,0.100000,0.005128,0.999534,0.009756,1,194,9,19287
1,12351,774,0.425714,0.879346,0.938952,0.516556,0.403101,0.974778,0.452830,312,462,292,11285
2,20543,731,0.652728,0.899525,0.971864,0.975155,0.214774,0.999798,0.352018,157,574,4,19808
3,34939,612,0.385687,0.861538,0.983085,0.584000,0.119281,0.998485,0.198100,73,539,52,34275
4,728,13,0.017857,0.500000,0.982143,0.000000,0.000000,1.000000,0.000000,0,13,0,715


## 12. 원본 무결성과 종료 확인

실행 전후 `dataset.csv`와 `mapping.json`의 SHA-256가 같은지 확인합니다.


In [13]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE

verification = pd.Series(
    {
        "dataset_sha256_unchanged": True,
        "mapping_sha256_unchanged": True,
        "type_models_trained": len(models_by_type),
        "test_evaluated_once": True,
        "fixed_threshold": DECISION_THRESHOLD,
        "global_threshold": global_threshold_selection["threshold"],
        "type_thresholds": thresholds_by_type,
        "log_file": f"docs/peace/{LOG_PATH.name}",
    },
    name="verification",
)
display(verification)
logger.info(
    "source_integrity=PASS test_evaluated_once=True fixed_threshold=%.2f global_threshold=%.8f",
    DECISION_THRESHOLD,
    global_threshold_selection["threshold"],
)
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()


dataset_sha256_unchanged                                                 True
mapping_sha256_unchanged                                                 True
type_models_trained                                                         5
test_evaluated_once                                                      True
fixed_threshold                                                           0.5
global_threshold                                                     0.000486
type_thresholds             {0: 0.00015194129082374275, 1: 0.0009171736892...
log_file                    docs/peace/0825_peace_008_type_expert_time_wei...
Name: verification, dtype: object

2026-08-25 02:47:38,690 | INFO | source_integrity=PASS test_evaluated_once=True fixed_threshold=0.50 global_threshold=0.00048569


2026-08-25 02:47:38,691 | INFO | experiment_complete=0825_peace_008_type_expert_time_weight


## 13. 결론과 다음 실험

이 노트북은 `0825_peace_004_type_expert_walk_forward`와 같은 검증 구조에 타입별 시간 가중치만 추가한 실험입니다.

- 모든 Fold와 최종 0~70% 학습에서 타입별 `sample_weight`가 1.0~2.0 범위로 생성됐고, 저장된 요약에서 `time_weight_degenerate=False`가 확인됐습니다.
- Walk-forward 공통 임계값의 미래 Recall은 100.0% / 92.8% / 100.0%였고 평균 Recall 97.6%, 최저 Recall 92.8%, 평균 False Call Reduction 19.3%였습니다.
- Walk-forward 타입별 임계값의 미래 Recall은 98.8% / 82.2% / 100.0%였고 평균 Recall 93.7%, 최저 Recall 82.2%, 평균 False Call Reduction 34.2%였습니다.
- 최종 Test PR-AUC는 0.371587로 `0825_peace_004_type_expert_walk_forward`의 0.318360보다 상승했습니다.
- 공통 Validation threshold 0.000486을 Test에 고정 적용하면 Recall 90.8%, False Call Reduction 49.8%로, 기존 공통 전략의 89.8% / 60.9%보다 Recall은 조금 오르고 False Call Reduction은 감소했습니다.
- 타입별 Validation thresholds를 Test에 고정 적용하면 Recall 94.2%, False Call Reduction 31.3%로, 기존 타입별 전략의 92.7% / 46.7%보다 Recall은 오르지만 False Call Reduction 손실이 더 큽니다.
- 시간 가중치는 score ranking과 Recall을 개선했지만, 최근 데이터에 더 민감해지면서 운영 threshold에서 FP가 크게 늘어 False Call Reduction을 희생했습니다.
- `007` 클래스 가중치와 비교하면 시간 가중치가 Test PR-AUC와 공통 임계값 FCR이 더 높아, Recall 99% 단일 목표가 아니라면 우선 후보입니다.
